# 🏋️ Module 4.5: Exercises

Practice Model Registry workflows.

---

In [ ]:
import mlflow
import mlflow.sklearn
from mlflow import MlflowClient
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

mlflow.autolog(disable=True)
mlflow.set_experiment("04_exercises")
client = MlflowClient()

wine = load_wine()
X_train, X_test, y_train, y_test = train_test_split(
    wine.data, wine.target, test_size=0.2, random_state=42
)
print("✅ Ready!")

## Exercise 1: Register 3 Model Versions

Register a model called `"ex4-wine-model"` with 3 versions:
1. LogisticRegression (max_iter=1000)
2. RandomForest (n_estimators=100)
3. GradientBoosting (n_estimators=100)

Add a description to each version.

In [ ]:
# YOUR CODE HERE


In [ ]:
# ✅ SOLUTION
EX_MODEL = "ex4-wine-model"

configs = [
    ("logreg", LogisticRegression(max_iter=1000, random_state=42), "LogisticRegression baseline"),
    ("rf", RandomForestClassifier(n_estimators=100, random_state=42), "RandomForest with 100 trees"),
    ("gb", GradientBoostingClassifier(n_estimators=100, random_state=42), "GradientBoosting with 100 estimators"),
]

versions = []
for name, model, desc in configs:
    with mlflow.start_run(run_name=f"ex4_{name}"):
        model.fit(X_train, y_train)
        acc = accuracy_score(y_test, model.predict(X_test))
        mlflow.log_metric("accuracy", acc)
        result = mlflow.sklearn.log_model(model, "model", registered_model_name=EX_MODEL)
        versions.append(result.registered_model_version)
        client.update_model_version(EX_MODEL, result.registered_model_version, description=desc)
        print(f"  v{result.registered_model_version}: {name} (acc={acc:.4f})")

print(f"\n✅ Registered {len(versions)} versions!")

## Exercise 2: Promote Best to Champion

Find the version with the highest accuracy and set it as the `champion` alias. Set the second-best as `challenger`.

In [ ]:
# YOUR CODE HERE


In [ ]:
# ✅ SOLUTION
all_versions = client.search_model_versions(f"name='{EX_MODEL}'")

# Get accuracy for each version
version_scores = []
for v in all_versions:
    try:
        run = client.get_run(v.run_id)
        acc = run.data.metrics.get("accuracy", 0)
        version_scores.append((v.version, acc))
    except:
        pass

# Sort by accuracy
version_scores.sort(key=lambda x: x[1], reverse=True)

if len(version_scores) >= 2:
    best_version, best_acc = version_scores[0]
    second_version, second_acc = version_scores[1]
    
    client.set_registered_model_alias(EX_MODEL, "champion", best_version)
    client.set_registered_model_alias(EX_MODEL, "challenger", second_version)
    
    print(f"🏆 Champion: v{best_version} (acc={best_acc:.4f})")
    print(f"⚔️  Challenger: v{second_version} (acc={second_acc:.4f})")

## Exercise 3: Load and Predict with Champion

Load the champion model by alias and make predictions on the test set. Print the first 5 predictions vs actual values.

In [ ]:
# YOUR CODE HERE


In [ ]:
# ✅ SOLUTION
champion = mlflow.sklearn.load_model(f"models:/{EX_MODEL}@champion")
preds = champion.predict(X_test[:5])

print("🔮 Champion Model Predictions:")
for i, (p, a) in enumerate(zip(preds, y_test[:5])):
    match = "✅" if p == a else "❌"
    print(f"  Sample {i+1}: predicted={wine.target_names[p]}, actual={wine.target_names[a]} {match}")

---

## 🎓 Module 4 Complete!

You've mastered:
- ✅ Registering models with two methods
- ✅ Managing model versions with descriptions and tags
- ✅ Champion/Challenger workflow with aliases
- ✅ Loading models by alias for production use

**Next up**: Module 5 — MLFlow Models & Signatures →